In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-flat-kp2000kd50'  # ckpt = 20000
# exp_name = 'friction-walking-fractal-kp2000kd50-action_rate-0.01'  # ckpt = 20000
exp_name = 'friction-walking-part-kp2000kd50'
# exp_name = 'correct-walking-flat-kp2000kd50-resume'
ckpt = 1000

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.1, 2.0],
  'restitution': [0.0, 0.2],
  'kp': [15000.0, 25000.0],
  'kd': [40.0, 80.0]

In [10]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [11]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [12]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.5607, -0.2840, -0.4613, -0.1562, -0.7766,  0.3465, -0.4299, -0.0665,
         -0.0489,  0.8123, -1.1169, -0.5346]], device='cuda:0')
Scaled actions :  tensor([[ 0.5607, -0.2840, -0.4613, -0.1562, -0.7766,  0.3465, -0.4299, -0.0665,
         -0.0489,  0.8123, -1.1169, -0.5346]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05,  5.6075e-01, -2.8399e-01,
         -4.6130e-01, -1.5618e-01, -7.7661e-01,  3.4653e-01, -4.2987e-01,
         -6.6507e-02, -4.8930e-02,  8.1230e-01, -1.1169e+00, -5.3455e-01]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.4409,  0.5358,  0.7443, -0.2402, -0.8484, -0.1402,  0.5287,  0.7080,
          0.1723, -0.1708, -0.3644,  0.2014]], device='cuda:0')
Scaled actions :  tensor([[-0.4409,  0.5358,  0.7443, -0.2402, -0.8484, -0.1402,  0.5287,  0.7080,
          0.1723, -0.1708, -0.3644,  0.2014]], device='cuda:0')
obs :  tensor([[ 0.5049,  0.2536, -0.0181,  0.0061, -0.0125, -0.9999,  1.0000,  0.0000,
          0.0000,  0.0450, -0.0276, -0.0129,  0.0216, -0.0776,  0.0828, -0.0544,
         -0.0144, -0.0244,  0.0397, -0.1032, -0.0990,  0.4177, -0.2323, -0.0899,
          0.1525, -0.7206,  0.6110, -0.4307, -0.1126, -0.1918,  0.3120, -0.9406,
         -0.7480, -0.4409,  0.5358,  0.7443, -0.2402, -0.8484, -0.1402,  0.5287,
          0.7080,  0.1723, -0.1708, -0.3644,  0.2014]], device='cuda:0')
torques: [ 200.          -92.24520015 -200.         -200.         -200.
 -200.           97.80264937  106.2861248   200.          200.
 -200.          200.        ]
データ収集: step 3


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-1.0753, -0.6658, -0.3863, -0.2521,  0.4773, -0.1556,  0.8036,  1.0235,
         -0.2153, -1.4561,  1.5514,  1.4512]], device='cuda:0')
Scaled actions :  tensor([[-1.0753, -0.6658, -0.3863, -0.2521,  0.4773, -0.1556,  0.8036,  1.0235,
         -0.2153, -1.4561,  1.5514,  1.4512]], device='cuda:0')
obs :  tensor([[-6.4628e-02, -3.2487e-01, -8.3305e-02,  3.5232e-03, -2.0175e-02,
         -9.9979e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  7.0521e-02,
         -3.9163e-02,  6.2978e-03,  3.7502e-02, -2.9665e-01,  8.6636e-02,
         -8.5306e-02, -1.9186e-02, -2.0159e-02,  6.2420e-02, -2.0781e-01,
         -1.4173e-01, -9.6055e-02,  7.7497e-02,  2.3909e-01, -1.4824e-03,
         -1.1798e+00, -4.2883e-01,  6.9087e-02,  4.8056e-02,  2.0091e-01,
         -4.4551e-02, -3.4688e-01,  2.2283e-01, -1.0753e+00, -6.6578e-01,
         -3.8629e-01, -2.5206e-01,  4.7734e-01, -1.5560e-01,  8.0363e-01,
          1.0235e+00, -2.1525e-01, -1.4561e+00,  1.5514e+00,  1.4

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 1.0252,  0.9511,  0.0884,  0.9121,  0.7987,  0.4485, -1.0755, -0.7975,
          0.3871,  0.3157,  0.6292, -0.4717]], device='cuda:0')
Scaled actions :  tensor([[ 1.0252,  0.9511,  0.0884,  0.9121,  0.7987,  0.4485, -1.0755, -0.7975,
          0.3871,  0.3157,  0.6292, -0.4717]], device='cuda:0')
obs :  tensor([[-0.0535,  0.3647, -0.1037,  0.0063, -0.0179, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0022, -0.0314,  0.0224,  0.0323, -0.4253, -0.0045, -0.0208,
         -0.0068, -0.0036,  0.0356, -0.1684,  0.0096, -0.5250,  0.0077, -0.0584,
          0.0348, -0.1670, -0.3085,  0.5329,  0.0687, -0.0078, -0.2097,  0.6416,
          1.1940,  1.0252,  0.9511,  0.0884,  0.9121,  0.7987,  0.4485, -1.0755,
         -0.7975,  0.3871,  0.3157,  0.6292, -0.4717]], device='cuda:0')
torques: [-200. -200. -200. -200.  200.  200.  200.  200. -200. -200.  200.  200.]
データ収集: step 5


In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 1.0031, -1.2773, -0.3728, -1.5883,  0.4284,  0.4055, -1.2736,  1.2331,
          0.0043, -0.6400, -0.1163, -0.9179]], device='cuda:0')
Scaled actions :  tensor([[ 1.0031, -1.2773, -0.3728, -1.5883,  0.4284,  0.4055, -1.2736,  1.2331,
          0.0043, -0.6400, -0.1163, -0.9179]], device='cuda:0')
obs :  tensor([[-0.1131, -0.2592,  0.0718,  0.0053, -0.0147, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0515, -0.0278,  0.0283,  0.0590, -0.3529,  0.0408,  0.0518,
         -0.0049, -0.0125,  0.0521,  0.0171,  0.0932, -0.0521,  0.0295,  0.0723,
          0.2135,  0.7961,  0.6637,  0.2026,  0.0184, -0.1245,  0.2913,  0.6328,
         -0.1438,  1.0031, -1.2773, -0.3728, -1.5883,  0.4284,  0.4055, -1.2736,
          1.2331,  0.0043, -0.6400, -0.1163, -0.9179]], device='cuda:0')
torques: [ 200.          200.           59.27689015  200.          200.
  200.         -200.         -200.          200.          200.
   12.96292281 -200.        ]
データ収集: step 6


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-0.1860,  0.6826,  0.7096, -1.3179, -1.0065, -0.8324,  0.5134, -0.6627,
         -0.8794, -1.2786, -0.8215,  0.9032]], device='cuda:0')
Scaled actions :  tensor([[-0.1860,  0.6826,  0.7096, -1.3179, -1.0065, -0.8324,  0.5134, -0.6627,
         -0.8794, -1.2786, -0.8215,  0.9032]], device='cuda:0')
obs :  tensor([[-1.2371e-01, -5.9907e-02,  2.1512e-01, -7.4690e-04, -9.9680e-03,
         -9.9995e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -9.7991e-03,
         -2.9905e-02,  3.7329e-02,  8.3692e-02, -1.3183e-01,  1.7324e-01,
          3.1818e-02,  1.2351e-02, -2.4889e-02,  8.0663e-02,  3.8848e-02,
         -4.7084e-02,  4.1583e-01, -3.6899e-02,  2.3803e-02,  5.1627e-02,
          1.2101e+00,  5.1899e-01, -3.4605e-01,  1.4230e-01, -8.6539e-03,
          2.0920e-02, -2.7432e-01, -1.1587e+00, -1.8599e-01,  6.8261e-01,
          7.0956e-01, -1.3179e+00, -1.0065e+00, -8.3239e-01,  5.1336e-01,
         -6.6266e-01, -8.7937e-01, -1.2786e+00, -8.2154e-01,  9.0

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-1.0468, -0.7229, -1.2495,  0.4096, -0.9318, -0.4197,  1.4437,  0.4449,
          1.1238,  0.2452,  0.6380,  1.6493]], device='cuda:0')
Scaled actions :  tensor([[-1.0468, -0.7229, -1.2495,  0.4096, -0.9318, -0.4197,  1.4437,  0.4449,
          1.1238,  0.2452,  0.6380,  1.6493]], device='cuda:0')
obs :  tensor([[-7.2441e-02, -3.2626e-03, -1.2517e-02, -1.8379e-03, -6.1396e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.1017e-02,
         -2.7399e-02,  6.3427e-02,  5.9730e-02,  3.0206e-03,  1.6621e-01,
          2.4342e-02,  2.5502e-02, -2.7505e-02,  6.9260e-02, -1.1998e-01,
         -1.6714e-01, -5.8845e-02,  4.9405e-02,  2.1770e-01, -2.5969e-01,
          2.3637e-01, -4.8928e-01,  2.1704e-01,  1.3881e-03, -1.8130e-02,
         -1.2149e-01, -1.2195e+00, -1.4436e-01, -1.0468e+00, -7.2292e-01,
         -1.2495e+00,  4.0956e-01, -9.3179e-01, -4.1968e-01,  1.4437e+00,
          4.4488e-01,  1.1238e+00,  2.4521e-01,  6.3799e-01,  1.6

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.9383,  0.9734, -0.0858, -1.3541,  0.6266,  0.4115, -1.0136, -0.8676,
         -0.0595, -0.4370,  0.9227, -0.1853]], device='cuda:0')
Scaled actions :  tensor([[ 0.9383,  0.9734, -0.0858, -1.3541,  0.6266,  0.4115, -1.0136, -0.8676,
         -0.0595, -0.4370,  0.9227, -0.1853]], device='cuda:0')
obs :  tensor([[-0.1348, -0.0846,  0.2301, -0.0036, -0.0019, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0471, -0.0201,  0.0822,  0.0505, -0.0778,  0.0022,  0.1147,
          0.0276, -0.0315,  0.0601, -0.2594, -0.0888, -0.5744,  0.0137,  0.0131,
          0.1057, -0.8508, -0.9077,  0.6433,  0.0181, -0.0231,  0.0205, -0.2700,
          0.8297,  0.9383,  0.9734, -0.0858, -1.3541,  0.6266,  0.4115, -1.0136,
         -0.8676, -0.0595, -0.4370,  0.9227, -0.1853]], device='cuda:0')
torques: [-200.         -200.         -200.          200.         -200.
   53.77545646  200.          200.          200.          200.
  200.          200.        ]
データ収集: step 9


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[ 0.5099, -1.0295,  0.4064, -0.4696,  0.5139,  0.8011, -0.8604,  0.9337,
          0.3320,  0.7287,  0.6062, -0.7968]], device='cuda:0')
Scaled actions :  tensor([[ 0.5099, -1.0295,  0.4064, -0.4696,  0.5139,  0.8011, -0.8604,  0.9337,
          0.3320,  0.7287,  0.6062, -0.7968]], device='cuda:0')
obs :  tensor([[-0.1348,  0.3876,  0.3513,  0.0035,  0.0036, -1.0000,  1.0000,  0.0000,
          0.0000, -0.1083, -0.0109,  0.0692,  0.0543, -0.1389, -0.0716,  0.1922,
          0.0275, -0.0412,  0.0376, -0.2064, -0.0300, -0.0864,  0.0750, -0.1285,
         -0.0513,  0.1405,  0.0704,  0.1775, -0.0187, -0.0600, -0.2268,  0.7026,
         -0.1435,  0.5099, -1.0295,  0.4064, -0.4696,  0.5139,  0.8011, -0.8604,
          0.9337,  0.3320,  0.7287,  0.6062, -0.7968]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.          200.
  200.         -200.         -200.           20.12839533 -200.
  200.         -200.        ]
データ収集: step 10

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 1.0284,  0.4537, -1.0022,  0.8137, -0.1291, -0.0458, -1.1720, -0.5519,
          0.2898,  0.2576, -1.1987,  0.9852]], device='cuda:0')
Scaled actions :  tensor([[ 1.0284,  0.4537, -1.0022,  0.8137, -0.1291, -0.0458, -1.1720, -0.5519,
          0.2898,  0.2576, -1.1987,  0.9852]], device='cuda:0')
obs :  tensor([[ 0.1516, -0.4107,  0.3064,  0.0014,  0.0046, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0698, -0.0144,  0.0859,  0.0177, -0.0308,  0.0397,  0.1568,
          0.0423, -0.0423,  0.0290,  0.0028, -0.1412,  0.4014, -0.1299,  0.1185,
          0.0493,  0.0051,  0.5157, -0.4462,  0.1064,  0.0984,  0.0325,  1.1962,
         -0.8457,  1.0284,  0.4537, -1.0022,  0.8137, -0.1291, -0.0458, -1.1720,
         -0.5519,  0.2898,  0.2576, -1.1987,  0.9852]], device='cuda:0')
torques: [ 200.         -200.          200.         -200.          200.
  200.         -200.          200.          200.          200.
  109.57619774 -200.        ]
データ収集: step 1

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[-0.4980, -0.8356, -0.3582, -1.8855,  0.1009, -0.7170,  0.9228,  1.3761,
         -0.3896, -1.0821, -1.3928,  0.3047]], device='cuda:0')
Scaled actions :  tensor([[-0.4980, -0.8356, -0.3582, -1.8855,  0.1009, -0.7170,  0.9228,  1.3761,
         -0.3896, -1.0821, -1.3928,  0.3047]], device='cuda:0')
obs :  tensor([[ 6.1668e-02, -4.0175e-01,  5.2305e-01, -1.4503e-02,  6.4160e-04,
         -9.9989e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  5.9166e-02,
         -2.6015e-02,  8.9757e-02,  6.2646e-02, -6.4065e-02,  4.7882e-02,
          1.5177e-02,  6.1291e-02, -3.4308e-02,  7.1940e-02,  5.1398e-02,
         -1.7741e-01,  8.4179e-01, -2.7722e-04, -6.2185e-02,  3.6850e-01,
         -1.6367e-01, -1.7219e-01, -9.1529e-01,  7.4920e-02,  6.1306e-02,
          2.6878e-01, -3.0292e-01,  2.9004e-01, -4.9804e-01, -8.3563e-01,
         -3.5825e-01, -1.8855e+00,  1.0088e-01, -7.1700e-01,  9.2279e-01,
          1.3761e+00, -3.8960e-01, -1.0821e+00, -1.3928e+00,  3.

In [39]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=3.678, Scaled action max=3.678
Step 1/10, Total steps: 172
steps: 172
actions : tensor([[ 2.0536, -2.7101, -0.2954, -2.5072, -3.8720,  3.6780, -1.0193,  3.4939,
         -2.6503,  1.6118,  1.6841, -2.9249]], device='cuda:0')
target_dof_pos: tensor([[ 1.8036,  1.3823, -0.6225,  0.3383, -3.7856,  2.8135,  0.0202, -0.9644,
         -0.5389,  2.5447, -1.1752, -1.6127]], device='cuda:0')
Step 1: Original action max=6.468, Scaled action max=6.468
Step 2: Original action max=6.396, Scaled action max=6.396
データ収集完了: 10 steps collected with action_scale=1.0


In [40]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [41]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
